In [ ]:
clc;
clear;

N = 1000;

arrival_min = 5;
arrival_max = 50;
service_min = 1;
service_max = 10;

arrival_rate = zeros(N,1);
service_time = zeros(N,1);
sim_output = zeros(N,1);

for i = 1:N
    arrival_rate(i) = arrival_min + (arrival_max-arrival_min)*rand();
    service_time(i) = service_min + (service_max-service_min)*rand();
    noise = randn()*0.5;
    sim_output(i) = 0.05*(arrival_rate(i)^2) + 2*service_time(i) + noise;
end

data = [arrival_rate service_time sim_output];

X = data(:,1:2);
Y = data(:,3);

ntrain = round(0.8*N);
Xtrain = X(1:ntrain,:);
Ytrain = Y(1:ntrain);
Xtest = X(ntrain+1:end,:);
Ytest = Y(ntrain+1:end);

Ypred_mean = mean(Ytrain) * ones(length(Ytest),1);
mse_mean = mean((Ytest - Ypred_mean).^2);

Xtrain_lr = [ones(ntrain,1) Xtrain];
b = Xtrain_lr \ Ytrain;
Xtest_lr = [ones(size(Xtest,1),1) Xtest];
Ypred_lr = Xtest_lr * b;
mse_lr = mean((Ytest - Ypred_lr).^2);

Xtrain_p2 = [ones(ntrain,1) Xtrain(:,1) Xtrain(:,2) Xtrain(:,1).^2 Xtrain(:,2).^2];
b2 = Xtrain_p2 \ Ytrain;
Xtest_p2 = [ones(size(Xtest,1),1) Xtest(:,1) Xtest(:,2) Xtest(:,1).^2 Xtest(:,2).^2];
Ypred_p2 = Xtest_p2 * b2;
mse_p2 = mean((Ytest - Ypred_p2).^2);

Xtrain_p3 = [ones(ntrain,1) Xtrain(:,1) Xtrain(:,2) Xtrain(:,1).^2 Xtrain(:,2).^2 Xtrain(:,1).^3];
b3 = Xtrain_p3 \ Ytrain;
Xtest_p3 = [ones(size(Xtest,1),1) Xtest(:,1) Xtest(:,2) Xtest(:,1).^2 Xtest(:,2).^2 Xtest(:,1).^3];
Ypred_p3 = Xtest_p3 * b3;
mse_p3 = mean((Ytest - Ypred_p3).^2);

Xtrain_p4 = [ones(ntrain,1) Xtrain(:,1) Xtrain(:,2) Xtrain(:,1).^2 Xtrain(:,2).^2 Xtrain(:,1).^3 Xtrain(:,1).^4];
b4 = Xtrain_p4 \ Ytrain;
Xtest_p4 = [ones(size(Xtest,1),1) Xtest(:,1) Xtest(:,2) Xtest(:,1).^2 Xtest(:,2).^2 Xtest(:,1).^3 Xtest(:,1).^4];
Ypred_p4 = Xtest_p4 * b4;
mse_p4 = mean((Ytest - Ypred_p4).^2);

lambda = 1;
I = eye(size(Xtrain_lr,2));
b_ridge = (Xtrain_lr' * Xtrain_lr + lambda * I) \ (Xtrain_lr' * Ytrain);
Ypred_ridge = Xtest_lr * b_ridge;
mse_ridge = mean((Ytest - Ypred_ridge).^2);

k = 3;
Ypred_knn3 = zeros(length(Ytest),1);
for i = 1:length(Ytest)
    diff = Xtrain - repmat(Xtest(i,:), size(Xtrain,1), 1);
    d = sum(diff.^2, 2);
    [sd, idx] = sort(d);
    Ypred_knn3(i) = mean(Ytrain(idx(1:k)));
end
mse_knn3 = mean((Ytest - Ypred_knn3).^2);

k = 5;
Ypred_knn5 = zeros(length(Ytest),1);
for i = 1:length(Ytest)
    diff = Xtrain - repmat(Xtest(i,:), size(Xtrain,1), 1);
    d = sum(diff.^2, 2);
    [sd, idx] = sort(d);
    Ypred_knn5(i) = mean(Ytrain(idx(1:k)));
end
mse_knn5 = mean((Ytest - Ypred_knn5).^2);

k = 7;
Ypred_knn7 = zeros(length(Ytest),1);
for i = 1:length(Ytest)
    diff = Xtrain - repmat(Xtest(i,:), size(Xtrain,1), 1);
    d = sum(diff.^2, 2);
    [sd, idx] = sort(d);
    Ypred_knn7(i) = mean(Ytrain(idx(1:k)));
end
mse_knn7 = mean((Ytest - Ypred_knn7).^2);

W = diag(ones(ntrain,1));  % simple weights
b_w = (Xtrain_lr' * W * Xtrain_lr) \ (Xtrain_lr' * W * Ytrain);
Ypred_w = Xtest_lr * b_w;
mse_w = mean((Ytest - Ypred_w).^2);

disp(' ');
disp('Model Comparison (Lower MSE is Better)');
disp('--------------------------------------');
fprintf('1. Mean Baseline           : %f\n', mse_mean);
fprintf('2. Linear Regression       : %f\n', mse_lr);
fprintf('3. Poly Degree 2           : %f\n', mse_p2);
fprintf('4. Poly Degree 3           : %f\n', mse_p3);
fprintf('5. Poly Degree 4           : %f\n', mse_p4);
fprintf('6. Ridge Regression        : %f\n', mse_ridge);
fprintf('7. KNN (k=3)               : %f\n', mse_knn3);
fprintf('8. KNN (k=5)               : %f\n', mse_knn5);
fprintf('9. KNN (k=7)               : %f\n', mse_knn7);
fprintf('10. Weighted Linear Reg    : %f\n', mse_w);
